In [1]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
from src.train import load_xy, split_train_test

X, y = load_xy()
X_train, X_test, y_train, y_test = split_train_test(X, y)

summary = pd.DataFrame({
    "nguồn": y.value_counts().sort_index(),
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
})
summary.index = ["Not CKD (0)", "CKD (1)"]
summary["tỷ lệ nguồn (%)"] = (summary["nguồn"] / len(y) * 100).round(1)
summary


,nguồn,train,test,tỷ lệ nguồn (%)
Not CKD (0),150,120,30,37.5
CKD (1),250,200,50,62.5


In [2]:
from src.config import NUMERIC_COLUMNS, CODED_COLUMNS, CATEGORY_VALUES

structure = pd.DataFrame([
    {"nhánh": "num — 11 cột số",
     "cột": ", ".join(NUMERIC_COLUMNS),
     "xử lý": "SimpleImputer(median) → StandardScaler\n"
              "(trung vị chịu cực trị tốt; chuẩn hóa để KNN/LogReg không bị cột to áp đảo)"},
    {"nhánh": "coded — 3 cột thứ bậc sg/al/su",
     "cột": ", ".join(CODED_COLUMNS),
     "xử lý": "ordinal: median + scale  |  onehot: mốt + OneHot\n"
              "(hai cách này được SO SÁNH BẰNG CV ở notebook 03, không chọn bằng mắt)"},
    {"nhánh": "cat — 10 cột phân loại",
     "cột": ", ".join(CATEGORY_VALUES),
     "xử lý": "SimpleImputer(most_frequent) → OneHotEncoder(handle_unknown='ignore')\n"
              "(mốt = giá trị nhiều nhất; handle_unknown để app không crash với giá trị lạ)"},
])
structure


,nhánh,cột,xử lý
0,num — 11 cột số,"age, bp, bgr, bu, sc, sod, pot, hemo, pcv, wc, rc",SimpleImputer(median) → StandardScaler\n(trung...
1,coded — 3 cột thứ bậc sg/al/su,"sg, al, su",ordinal: median + scale | onehot: mốt + OneH...
2,cat — 10 cột phân loại,"rbc, pc, pcc, ba, htn, dm, cad, appet, pe, ane",SimpleImputer(most_frequent) → OneHotEncoder(h...


In [3]:
from sklearn.linear_model import LogisticRegression
from src.train import make_pipeline

pipe = make_pipeline(LogisticRegression(max_iter=1000, random_state=42))
pipe.fit(X_train, y_train)

for name, step in pipe.named_steps.items():
    print(f"  {name:10s} → {type(step).__name__}")

age_median = pipe.named_steps["prepare"].named_transformers_["num"] \
    .named_steps["impute"].statistics_[0]
print(f"\n  Imputer điền 'age' bằng trung vị TRAIN = {age_median}")
print("  → Nếu lỡ fit trên toàn bộ 400 dòng, con số này đã 'thấy' dữ liệu test = rò rỉ!")


  clean      → FunctionTransformer
  prepare    → ColumnTransformer
  clf        → LogisticRegression

  Imputer điền 'age' bằng trung vị TRAIN = 54.0
  → Nếu lỡ fit trên toàn bộ 400 dòng, con số này đã 'thấy' dữ liệu test = rò rỉ!


In [4]:
from IPython.display import display

sample = X_test.head(5)
preview = pd.DataFrame({
    "id": sample["id"].values,
    "dự đoán (1=CKD)": pipe.predict(sample),
    "xác suất CKD": pipe.predict_proba(sample)[:, 1].round(3),
})
display(preview)

probe = X_test.head(3).copy().astype(object)
probe.iloc[0] = None
before = probe.copy(deep=True)
pipe.predict_proba(probe)
from pandas.testing import assert_frame_equal
assert_frame_equal(before, probe)
print("✓ Dữ liệu đầu vào không bị Pipeline sửa tại chỗ")
print("✓ Dòng toàn ô thiếu vẫn dự đoán được (coi như giá trị thiếu)")


,id,dự đoán (1=CKD),xác suất CKD
0,342,0,0.003
1,204,1,1.000
2,233,1,0.992
3,366,0,0.004
4,120,1,0.999


✓ Dữ liệu đầu vào không bị Pipeline sửa tại chỗ
✓ Dòng toàn ô thiếu vẫn dự đoán được (coi như giá trị thiếu)
